<a href="https://colab.research.google.com/github/srujith2006/ObjectDetection/blob/main/C_videoAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics
!pip install pennylane

In [1]:
from google.colab import files

uploaded = files.upload()

Saving Injured people detection.v1i.yolov8.zip to Injured people detection.v1i.yolov8.zip


In [3]:
import zipfile

zip_ref = zipfile.ZipFile(
    "Injured people detection.v1i.yolov8.zip",
    "r"
)

zip_ref.extractall("/content/")
zip_ref.close()

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
import shutil
import yaml # Added for parsing data.yaml

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader

# -----------------------------
# Dataset Preprocessing for ImageFolder
# -----------------------------

dataset_root = "/content/"
yolo_train_images = os.path.join(dataset_root, "train", "images")
yolo_train_labels = os.path.join(dataset_root, "train", "labels")
processed_train_root = os.path.join(dataset_root, "processed_dataset", "train")
data_yaml_path = os.path.join(dataset_root, "data.yaml")

# 1. Load data.yaml to get class names
class_names = []
try:
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
        class_names = data_config.get('names', [])
    print(f"Loaded class names from data.yaml: {class_names}")
except FileNotFoundError:
    print(f"Warning: data.yaml not found at {data_yaml_path}. Using fallback class names.")
    # Fallback to generic class names if data.yaml is missing or unreadable
    class_names = ["class_0", "class_1"] # Assuming a binary classification

if not class_names:
    print("Warning: No class names found. Using generic 'class_0', 'class_1'.")
    class_names = ["class_0", "class_1"]

# Determine indices for 'injured' and 'safe' classes
injured_class_idx = -1
safe_class_idx = -1

if 'injured' in class_names:
    injured_class_idx = class_names.index('injured')
else:
    # Assuming class 0 is 'injured' if not explicitly named
    injured_class_idx = 0
    if len(class_names) > injured_class_idx and class_names[injured_class_idx] != 'injured':
        print(f"Warning: 'injured' class not found in data.yaml names. Assuming class ID {injured_class_idx} is 'injured'.")

if 'safe' in class_names:
    safe_class_idx = class_names.index('safe')
else:
    # Assuming class 1 is 'safe' if not explicitly named, and if it exists
    if len(class_names) > 1:
        safe_class_idx = 1
    else:
        safe_class_idx = 0 # Fallback if only one class or less (may cause issues if only 'injured' exists)
    if len(class_names) > safe_class_idx and class_names[safe_class_idx] != 'safe':
        print(f"Warning: 'safe' class not found in data.yaml names. Assuming class ID {safe_class_idx} is 'safe'.")

# Create target directories for processed dataset
for class_name in class_names:
    os.makedirs(os.path.join(processed_train_root, class_name), exist_ok=True)
print(f"Created processed dataset directories under {processed_train_root}")

# 2. Process 'train' split
if not os.path.exists(yolo_train_images):
    raise FileNotFoundError(f"YOLO train images directory not found at {yolo_train_images}. Please check extraction.")

image_files = [f for f in os.listdir(yolo_train_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"Found {len(image_files)} image files in {yolo_train_images} for processing.")

for img_file in image_files:
    img_name, img_ext = os.path.splitext(img_file)
    label_file = img_name + ".txt"
    label_path = os.path.join(yolo_train_labels, label_file)
    image_path = os.path.join(yolo_train_images, img_file)

    assigned_class_name = None

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()

        is_injured_in_image = False
        for line in lines:
            parts = line.strip().split()
            if parts and int(parts[0]) == injured_class_idx: # Check if the 'injured' class is present
                is_injured_in_image = True
                break

        if is_injured_in_image:
            assigned_class_name = class_names[injured_class_idx]
        else:
            # If no injured objects, assign to 'safe'
            assigned_class_name = class_names[safe_class_idx]
    else:
        # If no label file, assume 'safe' (no detected objects = safe)
        assigned_class_name = class_names[safe_class_idx]

    if assigned_class_name:
        target_dir = os.path.join(processed_train_root, assigned_class_name)
        shutil.copy(image_path, target_dir)
    else:
        print(f"Warning: Could not assign class for {img_file}, skipping.")

print("Dataset transformation completed.")

# -----------------------------
# Image Transform
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# -----------------------------
# Load Dataset
# -----------------------------
dataset = datasets.ImageFolder(
    processed_train_root, # Changed path to the processed dataset
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

# -----------------------------
# Load Pretrained MobileNet
# -----------------------------
model = models.mobilenet_v2(pretrained=True)

# Modify Final Layer
model.classifier[1] = nn.Linear(1280, 2)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(5):

    running_loss = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}, Loss: {running_loss}"
    )

# Save Model
torch.save(
    model.state_dict(),
    "injury_model.pth"
)

print("Training Completed")

Loaded class names from data.yaml: ['Injured', 'Uninjured']
Created processed dataset directories under /content/processed_dataset/train
Found 113 image files in /content/train/images for processing.
Dataset transformation completed.
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 13.6M/13.6M [00:00<00:00, 110MB/s] 


Epoch 1, Loss: 9.667112443596125
Epoch 2, Loss: 6.516516864299774
Epoch 3, Loss: 5.6426221169531345
Epoch 4, Loss: 3.9768451247364283
Epoch 5, Loss: 1.2761854594573379
Training Completed


In [10]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.9 MB/s eta 0:00:00


In [9]:
uploaded = files.upload()

KeyboardInterrupt: 

In [11]:
import cv2
import torch
import torch.nn as nn

from ultralytics import YOLO

from torchvision import transforms
from torchvision import models

from PIL import Image

# -----------------------------------
# Load PRETRAINED YOLOv8
# -----------------------------------
detector = YOLO("yolov8n.pt")

# -----------------------------------
# Load Injury Classifier
# -----------------------------------
classifier = models.mobilenet_v2(
    pretrained=False
)

classifier.classifier[1] = nn.Linear(
    1280,
    2
)

classifier.load_state_dict(
    torch.load("injury_model.pth")
)

classifier.eval()

# -----------------------------------
# Transform
# -----------------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# -----------------------------------
# Open Video
# -----------------------------------
cap = cv2.VideoCapture(
    "/content/seedvideo.mp4"
)

# Get Width and Height
frame_width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

frame_height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

fps = int(
    cap.get(cv2.CAP_PROP_FPS)
)

# -----------------------------------
# Output Video
# -----------------------------------
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    "output.mp4",
    fourcc,
    fps,
    (frame_width, frame_height)
)

# -----------------------------------
# Process Video
# -----------------------------------
while True:

    ret, frame = cap.read()

    if not ret:
        break

    # YOLO Detection
    results = detector(frame)

    for r in results:

        boxes = r.boxes

        for box in boxes:

            cls = int(box.cls[0])

            # Detect ONLY persons
            if cls != 0:
                continue

            x1, y1, x2, y2 = map(
                int,
                box.xyxy[0]
            )

            conf = float(box.conf[0])

            # Crop Person
            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            # Convert to PIL
            img = Image.fromarray(crop)

            # Transform
            img = transform(img)

            img = img.unsqueeze(0)

            # Injury Prediction
            with torch.no_grad():

                output = classifier(img)

                pred = torch.argmax(output, 1)

            label = (
                "Injured"
                if pred == 0
                else "Safe"
            )

            # Width and Height
            width = x2 - x1
            height = y2 - y1

            # Draw Bounding Box
            cv2.rectangle(
                frame,
                (x1,y1),
                (x2,y2),
                (0,255,0),
                2
            )

            # Display Text
            text = (
                f"Survivor | "
                f"{label} | "
                f"W:{width} "
                f"H:{height}"
            )

            cv2.putText(
                frame,
                text,
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,0,255),
                2
            )

    # Write Frame
    out.write(frame)

print("Processing Completed")

cap.release()
out.release()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Streaming output truncated to the last 5000 lines.
Speed: 2.3ms preprocess, 112.7ms inference, 0.8ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 1 train, 111.4ms
Speed: 2.9ms preprocess, 111.4ms inference, 1.0ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 (no detections), 126.1ms
Speed: 3.0ms preprocess, 126.1ms inference, 0.8ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 (no detections), 111.8ms
Speed: 3.2ms preprocess, 111.8ms inference, 0.7ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 (no detections), 112.0ms
Speed: 3.2ms preprocess, 112.0ms inference, 0.7ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 (no detections), 122.1ms
Speed: 3.1ms preprocess, 122.1ms inference, 0.7ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 1 train, 116.6ms
Speed: 3.2ms preprocess, 116.6ms inference, 1.1ms postprocess per image at shape (1, 3, 288, 640)

0: 288x640 1 train, 112.1ms
Speed: 3.9ms preprocess, 112

In [12]:
from google.colab import files

files.download("output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>